# Deepfake Video Detection — Complete Corrected Google Colab Notebook

This notebook uses Google Drive, finds video folders, extracts frames, trains an EfficientNetB0 frame classifier, evaluates whole videos, and exports a TensorFlow Lite model for Android.

## Important dataset note
Your Drive screenshot showed:
`Deepfake_Detection_and_Generation_Challenge_Blue_Team/test_sample_videos`
with 400 MP4 files. A folder named `test_sample_videos` in a deepfake detection challenge dataset is usually an **unlabeled test set**. The notebook will find it for inference, but it will **not train a supervised model from those files alone**. To train, you must have labeled `Real` and `Fake` videos. The notebook below automatically searches the project folder for labeled training folders. If it cannot find them, it stops and explains what to do rather than inventing labels.

The uploaded `deepfake_video_model.tflite` is not assumed compatible; its input/output signature and training data have not been verified. This notebook creates a new frame model with a documented contract.

## 1. Install / import dependencies

In [ ]:
!pip -q install opencv-python-headless scikit-learn

import os, random, math, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
FRAMES_PER_VIDEO = 12
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FINE = 5
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpeg", ".mpg"}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Mount Drive and locate your existing dataset

In [ ]:
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Deepfake_Detection_and_Generation_Challenge_Blue_Team"
TEST_VIDEO_DIR = os.path.join(PROJECT_DIR, "test_sample_videos")

print("Project folder exists:", os.path.isdir(PROJECT_DIR))
print("Test video folder exists:", os.path.isdir(TEST_VIDEO_DIR))
if os.path.isdir(TEST_VIDEO_DIR):
    test_files = [f for f in os.listdir(TEST_VIDEO_DIR) if Path(f).suffix.lower() in VIDEO_EXTS]
    print("Unlabeled/test videos found:", len(test_files))
    print("Examples:", test_files[:10])

# Search for folders with Real/Fake-like names under the project.
# This does not label the files in test_sample_videos.
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        "Project folder not found. Check the exact name in MyDrive and update PROJECT_DIR."
    )

## 3. Find labeled videos for training

Accepted layouts include:
```
project/
  train/
    Real/ video files...
    Fake/ video files...
```
or
```
project/
  Real/ video files...
  Fake/ video files...
```
Folder names may also contain `original`, `authentic`, `deepfake`, or `manipulated`.

The notebook intentionally does not infer labels from random filenames in `test_sample_videos`.

In [ ]:
def label_from_path(path):
    parts = [p.lower() for p in Path(path).parts]
    # Use path components only, not arbitrary filename text.
    fake_names = {"fake", "fakes", "deepfake", "deepfakes", "manipulated", "manipulated_videos"}
    real_names = {"real", "reals", "original", "originals", "authentic", "real_videos"}
    for part in reversed(parts[:-1]):
        if part in fake_names:
            return 0
        if part in real_names:
            return 1
    # Allow class folder names like FakeVideos and RealVideos.
    for part in reversed(parts[:-1]):
        if "fake" in part:
            return 0
        if "real" in part or "original" in part or "authentic" in part:
            return 1
    return None

records = []
for root, dirs, files in os.walk(PROJECT_DIR):
    # Skip known unlabeled test folder so it is not accidentally included in training.
    if os.path.basename(root).lower() == "test_sample_videos":
        dirs[:] = []
        continue
    for filename in files:
        if Path(filename).suffix.lower() in VIDEO_EXTS:
            full_path = os.path.join(root, filename)
            label = label_from_path(full_path)
            if label is not None:
                records.append({"path": full_path, "label": int(label)})

videos = pd.DataFrame(records).drop_duplicates("path") if records else pd.DataFrame(columns=["path", "label"])
print("Labeled videos discovered:", len(videos))
if len(videos):
    print(videos["label"].map({0:"FAKE", 1:"REAL"}).value_counts())
    display(videos.head())
else:
    print("\nNo labeled Real/Fake training videos were found.")
    print("Your current test_sample_videos folder contains test videos, not verified labels.")
    print("Add a labeled training dataset to Drive, e.g. PROJECT_DIR/train/Real and PROJECT_DIR/train/Fake.")

## 4. Validate dataset and split by video

In [ ]:
if len(videos) < 10 or videos["label"].nunique() != 2:
    raise ValueError(
        "Cannot train yet: need at least 10 labeled videos and both Real and Fake classes. "
        "The test_sample_videos folder is excluded because its labels are not provided."
    )

trainval_df, test_df = train_test_split(
    videos, test_size=0.15, random_state=SEED, stratify=videos["label"]
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.1765, random_state=SEED, stratify=trainval_df["label"]
)
print("Train / validation / test video counts:", len(train_df), len(val_df), len(test_df))

## 5. Extract evenly spaced frames from each labeled video

In [ ]:
FRAME_DIR = "/content/deepfake_video_frames"
if os.path.exists(FRAME_DIR):
    shutil.rmtree(FRAME_DIR)
os.makedirs(FRAME_DIR, exist_ok=True)

def extract_video_frames(video_path, count=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if not cap.isOpened() or total <= 0:
        cap.release()
        return frames
    indices = np.linspace(0, total - 1, num=min(count, total), dtype=int)
    for idx in sorted(set(indices.tolist())):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok and frame is not None:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            frames.append(frame)
    cap.release()
    return frames

def frames_from_split(df, split_name):
    rows, bad = [], []
    for row in df.itertuples(index=False):
        frames = extract_video_frames(row.path)
        if not frames:
            bad.append(row.path)
            continue
        for i, frame in enumerate(frames):
            out_dir = os.path.join(FRAME_DIR, split_name, str(row.label))
            os.makedirs(out_dir, exist_ok=True)
            frame_path = os.path.join(out_dir, f"{Path(row.path).stem}_{i:03d}.jpg")
            Image.fromarray(frame).save(frame_path, quality=95)
            rows.append({"path": frame_path, "label": int(row.label), "video_path": row.path})
    print(split_name, "frames:", len(rows), "| unreadable videos:", len(bad))
    return pd.DataFrame(rows)

train_frames = frames_from_split(train_df, "train")
val_frames = frames_from_split(val_df, "val")
test_frames = frames_from_split(test_df, "test")
if min(len(train_frames), len(val_frames), len(test_frames)) == 0:
    raise ValueError("No frames extracted for one split. Check video files and dataset size.")

## 6. Build TensorFlow datasets

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_frame(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)  # 0..255; EfficientNetB0 contains rescaling
    return image, tf.cast(label, tf.float32)

def make_ds(frame_df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame_df["path"].astype(str).values,
        frame_df["label"].astype(np.float32).values
    ))
    if training:
        ds = ds.shuffle(min(len(frame_df), 10000), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_frame, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_frames, True)
val_ds = make_ds(val_frames)
test_ds = make_ds(test_frames)
weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1]), y=train_frames["label"].values
)
class_weights = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", class_weights)

## 7. Train EfficientNetB0 frame classifier

In [ ]:
from tensorflow.keras import layers, Model

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.08),
], name="augmentation")

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="rgb_0_255")
x = augment(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(1, activation="sigmoid", name="p_real")(x)
model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")]
)
OUT_DIR = os.path.join(PROJECT_DIR, "DeepfakeVideoModelOutput")
os.makedirs(OUT_DIR, exist_ok=True)
best_path = os.path.join(OUT_DIR, "best_video_frame_model.keras")
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(best_path, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1)
]
history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                     class_weight=class_weights, callbacks=callbacks)

## 8. Fine-tune selected layers

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")]
)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE,
                     class_weight=class_weights, callbacks=callbacks)
model = tf.keras.models.load_model(best_path)
print("Loaded best validation-AUC model:", best_path)

## 9. Evaluate on held-out test frames

In [ ]:
frame_probs = model.predict(test_ds, verbose=1).reshape(-1)
frame_y = test_frames["label"].values.astype(int)
frame_pred = (frame_probs >= 0.5).astype(int)
print("Frame accuracy:", accuracy_score(frame_y, frame_pred))
if len(np.unique(frame_y)) == 2:
    print("Frame ROC-AUC:", roc_auc_score(frame_y, frame_probs))
print(classification_report(frame_y, frame_pred, labels=[0,1],
                            target_names=["FAKE", "REAL"], zero_division=0))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(frame_y, frame_pred, labels=[0,1]))

## 10. Evaluate complete videos by aggregating frame scores

In [ ]:
def predict_video(video_path, aggregation="median"):
    frames = extract_video_frames(video_path, FRAMES_PER_VIDEO)
    if not frames:
        return np.nan, []
    scores = model.predict(np.asarray(frames, dtype=np.float32), verbose=0).reshape(-1)
    score = float(np.median(scores) if aggregation == "median" else np.mean(scores))
    return score, scores.tolist()

video_results = []
for row in test_df.itertuples(index=False):
    score, frame_scores = predict_video(row.path, "median")
    if not np.isnan(score):
        video_results.append({
            "video": row.path, "label": int(row.label),
            "p_real": score, "p_fake": 1.0-score, "frames_used": len(frame_scores)
        })

video_eval = pd.DataFrame(video_results)
video_eval["pred"] = (video_eval["p_real"] >= 0.5).astype(int)
print("Whole-video accuracy:", accuracy_score(video_eval["label"], video_eval["pred"]))
if video_eval["label"].nunique() == 2:
    print("Whole-video ROC-AUC:", roc_auc_score(video_eval["label"], video_eval["p_real"]))
print(classification_report(video_eval["label"], video_eval["pred"], labels=[0,1],
                            target_names=["FAKE", "REAL"], zero_division=0))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(video_eval["label"], video_eval["pred"], labels=[0,1]))
display(video_eval.head(20))

## 11. Export a float32 TFLite frame model

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
tflite_path = os.path.join(OUT_DIR, "deepfake_video_frame_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_bytes)
print("Saved TFLite:", tflite_path)
print("Model size (MB):", round(os.path.getsize(tflite_path)/(1024*1024), 2))

## 12. Verify Keras and TFLite predictions

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("Input:", inp["shape"], inp["dtype"])
print("Output:", out["shape"], out["dtype"])

sample_path = test_frames.iloc[0]["path"]
pil = Image.open(sample_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample = np.asarray(pil, dtype=np.float32)[None, ...]
keras_real = float(model.predict(sample, verbose=0).reshape(-1)[0])
interpreter.set_tensor(inp["index"], sample.astype(inp["dtype"]))
interpreter.invoke()
tflite_real = float(interpreter.get_tensor(out["index"]).reshape(-1)[0])
print(f"Keras P(REAL): {keras_real:.6f} ({keras_real*100:.2f}%)")
print(f"TFLite P(REAL): {tflite_real:.6f} ({tflite_real*100:.2f}%)")
print(f"Difference: {abs(keras_real-tflite_real)*100:.4f} percentage points")
plt.imshow(pil); plt.axis("off"); plt.show()

## 13. Save Android model contract

In [ ]:
metadata = {
    "input_shape": [1, 224, 224, 3],
    "input_dtype": "float32",
    "color": "RGB",
    "input_range": "0..255",
    "output": "single sigmoid P(REAL)",
    "fake_probability": "1 - P(REAL)",
    "video_aggregation": "median of evenly spaced frame P(REAL) values",
    "warning": "This is a frame classifier, not a temporal model accepting a whole video tensor."
}
with open(os.path.join(OUT_DIR, "android_model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))
print("Output folder:", OUT_DIR)

## Android integration checklist
- Copy `deepfake_video_frame_model.tflite` into `app/src/main/assets/`.
- Update the Android video classifier to load that exact filename.
- Sample multiple frames from each selected video (e.g. 12–24 evenly spaced frames).
- Resize each frame to 224×224, convert to RGB, and provide float32 values in 0..255.
- Each model output is `P(REAL)`; calculate `P(FAKE) = 1 - P(REAL)`.
- Aggregate frame scores (median is the notebook default).
- The new model is not a drop-in replacement for an unknown existing model unless the Android input/output code is updated to match.
- Evaluate with genuinely unseen, labeled videos. No notebook can guarantee a particular accuracy.